# 02o1: Extracting Data from a Single NotePlan File

This notebook demonstrates how to extract entities and relationships from a **single** NotePlan file using the graph builder agent. This is useful for testing extraction on a specific file or processing files one at a time.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup

All environment detection, Neo4j connection, and NotePlan directory configuration are handled in `00-import.ipynb`.

## Overview

This notebook focuses on extracting data from a **single NotePlan file**. It's perfect for:
- Testing extraction on a specific file
- Debugging extraction issues
- Understanding the extraction process step-by-step
- Processing files one at a time with detailed inspection

**Alternative Approaches:**
- [**02o2-extracting-data.ipynb**](./02o2-extracting-data.ipynb): Extract data from **multiple files** in bulk
- [**02o3-extracting-data-langchain.ipynb**](./02o3-extracting-data-langchain.ipynb): Alternative extraction method using LangChain's structured outputs

We'll:
1. Select a single NotePlan file to process
2. Use graph builder agent to extract entities and relationships
3. Display extracted data with detailed inspection
4. Save extracted data to disk (with caching support)


In [ ]:
# Run common imports and setup
# Note: nest_asyncio is automatically enabled in 00-import.ipynb
%run 00-import.ipynb

# Additional imports specific to this notebook
import asyncio
from knowledge_agents.agents.graph_builder_agent import run_graph_builder_agent

# NotePlan utilities
from knowledge_agents.notes.traversal import get_files_from_last_month
from knowledge_agents.notes.parser import read_noteplan_file
from knowledge_agents.notes.filter import should_skip_file

print("✅ Additional libraries imported")


✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded
✅ Repository components imported
🔍 Runtime detection: local
✅ Settings loaded:
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge
   Neo4j Username: neo4j
   Neo4j Password: ********
   LiteLLM Proxy Host: localhost

💡 To override settings, see Settings class docstring:
   help(Settings)  # or help(get_settings)
   # Quick examples:
   # settings = get_settings(neo4j_password='your_password')
   # settings = get_settings(runtime_env='container')

📁 NotePlan directory: /Users/omareid/Library/Containers/co.noteplan.NotePlan3/Data/Library/Application Support/co.noteplan.NotePlan3
   Directory exists: True

✅ Successfully connected to Neo4j
✅ Data manipulation libraries imported
✅ Additional libraries imported


## Select a Single NotePlan File

Choose a specific NotePlan file to extract data from. Note: `NOTEPLAN_DIR` is already set up in `00-import.ipynb` based on the runtime environment.


In [ ]:
# Option 1: Select a specific file by path
# Uncomment and modify the path below to target a specific file
# file_path = NOTEPLAN_DIR / "Calendar" / "2025-01-15.md"
# relative_path = "Calendar/2025-01-15.md"

# Option 2: Select the first file from recent files (for demo)
# Get NotePlan files from the last month
all_files = get_files_from_last_month(NOTEPLAN_DIR)
print(f"Found {len(all_files)} files in last month")

# Filter out files we should skip
valid_files = [(fp, mod_time) for fp, mod_time in all_files if not should_skip_file(fp)]
print(f"After filtering: {len(valid_files)} valid files")

# Select the first file for single-file processing
if valid_files:
    file_path, mod_time = valid_files[0]
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    print(f"\n📄 Selected file: {relative_path}")
    print(f"   Full path: {file_path}")
    print(f"   Modified: {mod_time}")
else:
    raise ValueError("No valid files found to process")


Found 176 files to process
After filtering: 176 files
Processing 5 files for this demo


## Extract Entities and Relationships

Use the graph builder agent to extract entities and relationships from the selected file.


In [ ]:
# Import utilities for data persistence and processing
from knowledge_agents.utils import (
    check_file_cached,
    get_data_dir,
    load_nodes_edges,
    process_file_with_sections_and_embeddings,
    save_nodes_edges,
    save_sections_embeddings,
)
from pathlib import Path

# Set up data directory (build/data/)
project_root = Path(os.getcwd())
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = get_data_dir(project_root / "build", NOTEPLAN_DIR)
print(f"📁 Data directory: {data_dir}")

# Check cache for this single file
use_cache = True  # Set to False to regenerate
is_cached = False
output = None
sections_with_embeddings = []

if use_cache:
    is_cached, nodes_edges_path, _ = check_file_cached(
        data_dir, relative_path, check_nodes_edges=True, check_sections_embeddings=False
    )
    if is_cached and nodes_edges_path:
        print(f"✅ File is cached: {relative_path}")
        cached_data = load_nodes_edges(nodes_edges_path)
        print(f"   Found {len(cached_data.get('entities', []))} entities, {len(cached_data.get('relationships', []))} relationships")
        print(f"   Cache location: {nodes_edges_path}")
        print("\n💡 To reprocess this file, set use_cache=False above")
    else:
        print(f"📝 File not cached, will process: {relative_path}")

# Process file if not cached
if not is_cached:
    print(f"\n🔄 Processing file: {relative_path}")
    print(f"   This may take a minute as the AI analyzes the content...")
    
    # Process file: extract nodes/edges and generate sections with embeddings
    # Note: Logging is automatically written to stdout.log and stderr.log files
    try:
        output, sections_with_embeddings = asyncio.run(
            process_file_with_sections_and_embeddings(
                file_path=file_path,
                relative_path=relative_path,
                dependencies=dependencies,
                data_dir=data_dir,
                generate_embeddings_flag=True,
                use_cache=False,  # Already checked above
            )
        )
        
        # Save nodes/edges if extraction succeeded
        if output:
            save_nodes_edges(data_dir, relative_path, output, source_file_path=file_path)
            print(f"\n✅ Successfully extracted:")
            print(f"   - {len(output.entities)} entities")
            print(f"   - {len(output.relationships)} relationships")
        else:
            print(f"\n❌ Failed to extract entities/relationships")
            print(f"   Check the stderr.log file for details: {data_dir / relative_path}")
        
        # Save sections with embeddings
        if sections_with_embeddings:
            save_sections_embeddings(data_dir, relative_path, sections_with_embeddings, source_file_path=file_path)
            print(f"   - {len(sections_with_embeddings)} sections with embeddings")
        
        # Note: stdout.log and stderr.log are automatically created by the logging system
        print(f"\n📋 Logs saved to:")
        print(f"   - stdout.log: {data_dir / relative_path / Path(relative_path).stem}_stdout.log")
        print(f"   - stderr.log: {data_dir / relative_path / Path(relative_path).stem}_stderr.log")
        
    except Exception as e:
        print(f"\n❌ Error processing {relative_path}: {e}")
        print(f"   Check the stderr.log file for details")
        # Logs are automatically saved even on exception
        raise

# Load output for display (either from cache or just processed)
if not output and is_cached:
    file_data_dir = data_dir / relative_path
    file_data_dir = file_data_dir.parent
    base_name = Path(relative_path).stem
    cached_path = file_data_dir / f"{base_name}_nodes_edges.json"
    if cached_path.exists():
        cached_data = load_nodes_edges(cached_path)
        # Convert cached data back to GraphBuilderAgentOutput format for display
        from knowledge_agents.types.graph import Entity, Relationship, GraphBuilderAgentOutput
        entities = [Entity(**e) for e in cached_data.get('entities', [])]
        relationships = [Relationship(**r) for r in cached_data.get('relationships', [])]
        output = GraphBuilderAgentOutput(entities=entities, relationships=relationships, insights=cached_data.get('insights', []))

Processing: np-out.log


OPENAI_API_KEY is not set, skipping trace export
Error in graph builder agent for np-out.log: Invalid JSON when parsing { "entities": [ { "name": "2025-03-01 16:09:59 +0000", "type": "Date", "properties": { } }, { "name": "v3.16 (1323)", "type": "AppVersion", "properties": { } }, { "name": "macOS (Version 15.3.1 (Build 24D70))", "type": "OS", "properties": { } }, { "name": "2025-03-01", "type": "Date", "properties": { } }, { "name": "2025-03-01 16:09:52 +0000", "type": "Date", "properties": { } }, { "name": "2025-03-01 16:10:07 +0000", "type": "Date", "properties": { } }, { "name": "DFF38DBF-2293-4D22-A43C-D6DF5CA1A303", "type": "RecordID", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { 

  ✅ Extracted 0 entities, 0 relationships
Processing: np-error.log


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 12 entities, 11 relationships
Processing: Filters/folders.views


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 8 entities, 5 relationships
Processing: Calendar/20250927.txt


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 4 entities, 5 relationships
Processing: Calendar/20251026.txt
  ✅ Extracted 0 entities, 0 relationships

✅ Processed 5 files successfully


OPENAI_API_KEY is not set, skipping trace export


## Display Extracted Data

View the extracted entities and relationships.


In [ ]:
# Display extracted entities and relationships for this single file
if output:
    print(f"📊 Extracted Data from: {relative_path}\n")
    
    # Display entities
    if output.entities:
        print(f"✅ Entities ({len(output.entities)}):")
        entities_data = []
        for entity in output.entities:
            entities_data.append({
                "name": entity.name,
                "type": entity.type,
                "properties": entity.properties
            })
        df_entities = pd.DataFrame(entities_data)
        print(df_entities.to_string(index=False))
        print()
    else:
        print("⚠️  No entities extracted")
    
    # Display relationships
    if output.relationships:
        print(f"🔗 Relationships ({len(output.relationships)}):")
        relationships_data = []
        for rel in output.relationships:
            relationships_data.append({
                "from": rel.from_entity,
                "type": rel.type,
                "to": rel.to_entity,
                "properties": rel.properties
            })
        df_relationships = pd.DataFrame(relationships_data)
        print(df_relationships.to_string(index=False))
        print()
    else:
        print("⚠️  No relationships extracted")
    
    # Display insights if available
    if output.insights:
        print(f"💡 Insights:")
        for insight in output.insights:
            print(f"   - {insight}")
else:
    print("❌ No data extracted. Check the logs for errors.")


## Next Steps

Now that data is extracted from a single file, you can:

**For Single File:**
- Review the extracted entities and relationships above
- Check the logs (stdout.log, stderr.log) for detailed processing information
- Modify the file selection above to process a different file

**For Bulk Processing:**
- [**02o2-extracting-data.ipynb**](./02o2-extracting-data.ipynb): Extract data from **multiple files** in bulk

**Continue with Data Storage:**
- [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb): Generate vector embeddings from NotePlan notes
- [**03-loading-data.ipynb**](./03-loading-data.ipynb): Load entities and relationships into Neo4j graph
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store vector embeddings in Neo4j
